<div align="center">

# Serve fse search with FastAPI

Answer many searches at once from one store, then from three stores on a shared drive.

</div>

## When do you need this?

A web service gets many search requests at the same time. Each search reads the rows of a store. fse already lets many threads read one store, and its native code releases Python's GIL while it works. The service has to make three choices:

- **Where the search runs.** A search blocks its thread. In an `async` service it must not run on the event loop. It runs in a worker thread.
- **How many searches run at once.** Without a limit, every request could start its own threads. One shared `fse.Pool` and one limiter keep the CPU count in check.
- **When the store changes.** A store that other processes still write can grow. The service opens a fresh view from time to time and swaps it in without stopping the searches that run.

This recipe is about correctness. It checks that parallel requests return the same bits as serial requests. It measures no time and gives no speed figures.

## The story in a table

| Step | What happens | Call |
|---|---|---|
| 1 | Write one store and a service file | `ShardSink`, `FastAPI` |
| 2 | Start the service with the limits set by environment variables | `uvicorn`, `PYTHON_CPU_COUNT` |
| 3 | Send many requests at once and compare them with serial requests | `httpx.AsyncClient` |
| 4 | Stop the service and wait for running searches | `lifespan` |
| 5 | Three writers write three stores in one folder | `encode(shard=(i, 3))` |
| 6 | The service reads the three stores as one view and opens it again from time to time | `VectorStore.open_view` |

## 1. Texts, a model and one store

We use news articles from the [AG News](https://huggingface.co/datasets/SetFit/ag_news) train split as documents, 120,000 of them, and 64 articles from its test split as queries.
The model is [POTION-base-8M](https://huggingface.co/minishlab/potion-base-8M).
The [shared setup](README.md#run-a-recipe) installs the pinned packages. This recipe also needs `fastapi`, `uvicorn` and `httpx` from `requirements.txt`.

The store keeps one `int64` document ID per row. Here the ID is the row of the source text.

In [1]:
import asyncio
import json
import os
import signal
import subprocess
import sys
import time
from importlib.metadata import version
from pathlib import Path

import httpx
import numpy as np
from _helpers import Recipe

import fse

recipe = Recipe()
work = recipe.work
print({name: version(name) for name in ("fastapi", "uvicorn", "httpx", "anyio", "starlette")})

documents, _ = recipe.dataset("AG News", "train")
queries, _ = recipe.dataset("AG News", "test")
queries = queries[:64]
encoder = recipe.encoder()

with fse.ShardSink(work / "whole", encoder, doc_ids=True) as sink:
    encoder.encode(documents, doc_ids=np.arange(len(documents)), sink=sink)
store = fse.VectorStore.open(work / "whole", encoder=encoder)
print("Rows:", len(store), "Queries:", len(queries))

Recipe run: 2026-11-08. Generated files are temporary.
Apple M1 Pro. macOS-27.0.1-arm64-arm-64bit-Mach-O. Python 3.13.16.
{'fastapi': '0.162.0', 'uvicorn': '0.54.0', 'httpx': '0.28.1', 'anyio': '4.15.1', 'starlette': '1.10.0'}


Rows: 120000 Queries: 64


## 2. The service

The file below is the whole service. Read it from top to bottom, because each part answers one question from above.

- **`lifespan`** runs once in each process, after the process has started. The process opens its own store there. A store object must never cross a `fork`, so a process manager that loads the application first and forks later (for example gunicorn with `--preload`) is not allowed.
- **`fse.Pool`** holds the worker threads. All requests share it. Its size comes from `usable_cpus()`, which follows `PYTHON_CPU_COUNT`.
- **`CapacityLimiter`** is the limiter of the service. Its size is the size of the pool. AnyIO's default limiter allows 40 threads, which is more than a small pool needs.
- **`anyio.to_thread.run_sync`** runs the blocking search in a worker thread and waits for it without blocking the event loop.
- **`mode="exact"`** makes the answer reproducible. The same query gives the same bits, however many searches run beside it.
- **The reopen loop** reads the list of stores again, opens a new view at a fixed distance and swaps it in. A search that began on the old view finishes on the old view. The service closes the old view when its last search ends.
- **The end of `lifespan`** stops the loop, waits for the running searches, closes the view and then closes the pool.

In [2]:
(work / "service.py").write_text('''
import logging
import os
from contextlib import asynccontextmanager
from functools import partial
from pathlib import Path

import anyio
from fastapi import FastAPI, Request
from pydantic import BaseModel, Field

import fse

log = logging.getLogger("uvicorn.error")


class Query(BaseModel):
    text: str
    k: int = Field(10, ge=1, le=100)


class Generation:
    """One opened store or view, and the requests that still use it."""

    def __init__(self, store, number):
        self.store, self.number = store, number
        self.rows = len(store)
        self.inflight = 0  # touched only on the event loop
        self.retired = False
        self.closed = False


class Service:
    def __init__(self):
        self.listing = Path(os.environ["FSE_STORE_LIST"])  # one store path per line
        self.interval = float(os.environ.get("FSE_REOPEN_SECONDS", "0"))
        self.encoder = fse.Encoder.from_pretrained(os.environ["FSE_MODEL"])
        self.pool = fse.Pool(start=True)  # usable_cpus() threads, or PYTHON_CPU_COUNT
        self.limiter = anyio.CapacityLimiter(self.pool.workers)
        self.running = 0
        self.number = 0
        self.current = None

    def open_stores(self):
        lines = self.listing.read_text().splitlines()  # read at every open
        paths = [Path(line.strip()) for line in lines if line.strip()]
        if len(paths) == 1:
            return fse.VectorStore.open(paths[0], encoder=self.encoder)
        return fse.VectorStore.open_view(paths, encoder=self.encoder)

    async def open_generation(self):
        store = await anyio.to_thread.run_sync(self.open_stores)
        self.number += 1
        return Generation(store, self.number)

    async def release(self, generation):
        if generation.retired and generation.inflight == 0 and not generation.closed:
            generation.closed = True
            with anyio.CancelScope(shield=True):
                await anyio.to_thread.run_sync(generation.store.close)

    async def reopen_loop(self):
        while True:
            await anyio.sleep(self.interval)
            try:
                fresh = await self.open_generation()
            except Exception as error:  # keep serving the view we have
                log.warning("reopen failed (%s), keeping the current view", type(error).__name__)
                continue
            old = self.current
            if fresh.rows == old.rows:
                fresh.retired = True
                await self.release(fresh)
                continue
            self.current = fresh  # one assignment: new requests see the new view
            old.retired = True
            await self.release(old)
            log.info("view %d has %d rows", fresh.number, fresh.rows)

    def search(self, generation, text, k):
        vectors = self.encoder.encode([text], pool=self.pool)
        found = generation.store.search(vectors, k, mode="exact", pool=self.pool)
        return {
            "view": generation.number,
            "rows": generation.rows,
            "ids": found.ids[0].tolist(),
            "doc_ids": found.doc_ids[0].tolist(),
            "scores": found.scores[0].tolist(),  # float32 values, exact in a float64
        }

    async def drain(self):
        while self.running:
            await anyio.sleep(0.05)

    async def close(self):
        try:
            if self.current is not None:
                self.current.retired = True
                await self.release(self.current)
        finally:
            self.pool.close()  # waits for running tasks


@asynccontextmanager
async def lifespan(app):
    service = Service()
    try:
        service.current = await service.open_generation()
        app.state.service = service
        log.info("opened %d rows, %d pool workers", service.current.rows, service.pool.workers)
        async with anyio.create_task_group() as tasks:
            if service.interval:
                tasks.start_soon(service.reopen_loop)
            yield
            tasks.cancel_scope.cancel()  # stop the reopen loop
    finally:  # also when the first open fails
        with anyio.CancelScope(shield=True):
            await service.drain()
            await service.close()
    log.info("all searches finished, store and pool closed")


app = FastAPI(lifespan=lifespan)


@app.get("/info")
async def info(request: Request):
    service = request.app.state.service
    return {"pid": os.getpid(), "view": service.current.number, "rows": service.current.rows,
            "pool_workers": service.pool.workers, "limiter": service.limiter.total_tokens}


@app.post("/search")
async def search(query: Query, request: Request):
    service = request.app.state.service
    generation = service.current  # the view this request will use, to the end
    generation.inflight += 1
    service.running += 1
    try:
        call = partial(service.search, generation, query.text, query.k)
        return await anyio.to_thread.run_sync(call, limiter=service.limiter)
    finally:
        generation.inflight -= 1
        service.running -= 1
        await service.release(generation)
''')
print((work / "service.py").stat().st_size, "bytes written")

5097 bytes written


## 3. Start it

The helper starts `uvicorn` with a given number of processes and sets the limits through environment variables.
**Processes times pool workers must not exceed `usable_cpus()`.** Otherwise the threads of all processes compete for the same CPUs. We give each process an equal share of the CPUs and pass it as `PYTHON_CPU_COUNT`.
The BLAS libraries of numpy may start threads of their own. fse does not limit them, so the service sets them to one before numpy is imported. The `mode="exact"` search does not use BLAS, but other code in your service may.

In [3]:
with fse.Pool() as pool:
    cpus = pool.workers  # usable_cpus(): affinity, container quota or PYTHON_CPU_COUNT
print("Usable CPUs:", cpus)


def serve(listing, *, processes, reopen=0.0, port):
    assert 1 <= processes <= cpus, "processes times pool workers must not exceed the CPUs"
    share = max(1, cpus // processes)
    env = {
        **os.environ,
        "FSE_MODEL": str(recipe.model_folder()),
        "FSE_STORE_LIST": str(listing),
        "FSE_REOPEN_SECONDS": str(reopen),
        "PYTHON_CPU_COUNT": str(share),
        "OMP_NUM_THREADS": "1",
        "OPENBLAS_NUM_THREADS": "1",
        "VECLIB_MAXIMUM_THREADS": "1",
    }
    log = open(work / f"service-{port}.log", "w")
    command = [sys.executable, "-m", "uvicorn", "service:app", "--host", "127.0.0.1",
               "--port", str(port), "--workers", str(processes)]
    process = subprocess.Popen(command, cwd=work, env=env, stdout=log, stderr=subprocess.STDOUT)
    for _ in range(300):
        try:
            if httpx.get(f"http://127.0.0.1:{port}/info").status_code == 200:
                return process, f"http://127.0.0.1:{port}", work / f"service-{port}.log"
        except httpx.TransportError:
            time.sleep(0.2)
    process.kill()
    raise RuntimeError((work / f"service-{port}.log").read_text())


def stop(process, log):
    process.send_signal(signal.SIGTERM)  # uvicorn runs the end of lifespan, then exits
    process.wait(timeout=60)
    return [line for line in log.read_text().splitlines() if "fse" in line or "view" in line or "searches" in line or "opened" in line]


(work / "whole.txt").write_text(str(work / "whole"))
process, url, log = serve(work / "whole.txt", processes=min(2, cpus), port=18001)
print(httpx.get(url + "/info").json())

Usable CPUs: 10


{'pid': 1992, 'view': 1, 'rows': 120000, 'pool_workers': 5, 'limiter': 5}


## 4. Parallel requests give the same bits

We send the 64 queries three ways and compare them bit for bit:

1. **Serial over HTTP**: one request after the other.
2. **Parallel over HTTP**: all 64 requests at once through one `httpx.AsyncClient`, 32 at a time.
3. **Direct**: one `store.search(..., mode="exact")` call in this notebook.

The scores travel as JSON numbers. A float32 value is exact as a float64, so converting the numbers back to float32 gives the bits that the service computed. We compare the bit patterns, not the values.

In [4]:
K = 10


def unpack(answers):
    ids = np.array([a["ids"] for a in answers], dtype=np.int64)
    doc_ids = np.array([a["doc_ids"] for a in answers], dtype=np.int64)
    scores = np.array([a["scores"] for a in answers], dtype=np.float32)
    return ids, doc_ids, scores


def serial(url):
    with httpx.Client(base_url=url, timeout=60) as client:
        return [client.post("/search", json={"text": q, "k": K}).raise_for_status().json() for q in queries]


async def parallel(url, width=32):
    gate = asyncio.Semaphore(width)
    async with httpx.AsyncClient(base_url=url, timeout=60) as client:
        async def one(q):
            async with gate:
                reply = await client.post("/search", json={"text": q, "k": K})
                return reply.raise_for_status().json()
        return await asyncio.gather(*(one(q) for q in queries))


def same_bits(a, b):
    return all(np.array_equal(x.view(np.uint32) if x.dtype == np.float32 else x,
                              y.view(np.uint32) if y.dtype == np.float32 else y) for x, y in zip(a, b, strict=True))


one_by_one = unpack(serial(url))
answers = await parallel(url)
at_once = unpack(answers)
found = store.search(encoder.encode(queries), K, mode="exact")
direct = (found.ids, found.doc_ids, found.scores)

print("parallel == serial:", same_bits(at_once, one_by_one))
print("parallel == direct:", same_bits(at_once, direct))
assert same_bits(at_once, one_by_one) and same_bits(at_once, direct)

parallel == serial: True
parallel == direct: True


## 5. Stop and wait

`SIGTERM` makes uvicorn stop accepting connections, finish the requests it has and run the end of `lifespan`. The service then closes the store and the pool. A store refuses `close` while a search still reads it, so closing last is how a service avoids that error.

In [5]:
for line in stop(process, log):
    print(line)

INFO:     opened 120000 rows, 5 pool workers
INFO:     opened 120000 rows, 5 pool workers
INFO:     all searches finished, store and pool closed
INFO:     all searches finished, store and pool closed


## 6. Several machines on one shared drive

Large clusters often have a fast network drive. Here is the simplest layout that uses it:

```
machine 1: writer 0 -> shared/store-0
machine 2: writer 1 -> shared/store-1      search service(s):
machine 3: writer 2 -> shared/store-2        VectorStore.open_view([store-0, store-1, store-2])
```

- **One writer for each store, never two in one store.** Two writers fight for the name of the next shard. Each writer calls `encode(..., shard=(i, 3))`, which gives writer `i` its own third of the source rows, cut at multiples of 16,384 rows.
- **`durability="full"`** is the default. A shard written this way carries no tag, so opening the store does not hash its data again. On a local file system it also makes a published shard durable once the sync of the store directory has succeeded. A network drive gives no such promise.
- **`open_view`** reads the three stores as one store, without copying them.
- **A growing store needs a new view.** Do not rely on `refresh` or on directory lists of a network drive, because clients can cache them. The service opens a new view at a fixed distance. Shards never change after they appear, so a search on the old view stays correct while the new view takes the next requests.
- **IDs on a growing view.** While a part is still written, the position of a row in the view can change. The document ID does not. Return `doc_ids` to your clients, as the service does.

**What this recipe does not claim.** A network drive is outside the crash and power-loss contract of fse, which covers local APFS, ext4 and XFS. This recipe also gives no speed figure: an exact search reads the mapped files, and on a network drive its speed depends on the page cache, on the network and on your drive. Measure that before you plan capacity.

We replay this on one machine. Three processes stand in for the machines, and a folder stands in for the drive.

In [6]:
(work / "writer.py").write_text('''
import json
import sys
from pathlib import Path

import numpy as np

import fse

part, parts, folder, texts, model = int(sys.argv[1]), int(sys.argv[2]), *map(Path, sys.argv[3:6])
documents = json.loads(texts.read_text())
encoder = fse.Encoder.from_pretrained(model)
with fse.ShardSink(folder / f"store-{part}", encoder, resume=True, doc_ids=True,
                   durability="full", shard_rows=16384) as sink:
    encoder.encode(documents, doc_ids=np.arange(len(documents)), sink=sink, shard=(part, parts))
''')
(work / "documents.json").write_text(json.dumps(documents))
drive = work / "drive"  # a folder in place of the network drive
drive.mkdir()
print("ready")

ready


Writers 0 and 1 run first. The third machine joins later. We start the service on the two finished stores with a list file, then start writer 2, and add its store to the list when it has finished.
The service reads the list and opens a new view every 0.1 seconds. We send requests all the time and check what must hold: no request fails during a swap, the row count never shrinks, it starts below the full count and it ends at the full count. The steps in between depend on timing.

In [7]:
def write(part):
    return subprocess.Popen([sys.executable, "writer.py", str(part), "3", str(drive), "documents.json",
                             str(recipe.model_folder())], cwd=work)


first = [write(0), write(1)]
assert [w.wait() for w in first] == [0, 0]
stores = [drive / f"store-{i}" for i in range(3)]
listing = work / "stores.txt"
listing.write_text("\n".join(str(s) for s in stores[:2]))
process, url, log = serve(listing, processes=1, reopen=0.1, port=18002)

seen, failures = [], 0
with httpx.Client(base_url=url, timeout=60) as client:
    seen.append(client.get("/info").json()["rows"])
    late = write(2)
    while late.poll() is None:  # searches go on while writer 2 works
        failures += client.post("/search", json={"text": queries[0], "k": K}).status_code != 200
    listing.write_text("\n".join(str(s) for s in stores))  # writer 2 is done: list its store
    for _ in range(100):
        time.sleep(0.1)
        failures += client.post("/search", json={"text": queries[0], "k": K}).status_code != 200
        seen.append(client.get("/info").json()["rows"])
        if seen[-1] == len(documents):
            break
print("Writer 2 exit code:", late.returncode)
print("Row counts seen:", sorted(set(seen)))
assert failures == 0 and seen == sorted(seen)
assert seen[0] < len(documents) == seen[-1]

Writer 2 exit code: 0
Row counts seen: [81920, 120000]


## 7. The view equals one store with the same rows

The writers wrote three stores. `whole` is one store written from the same texts in one run. A search over the view must give the same answer. For a view of all finished parts, the positions are the source rows, so even the `ids` agree.

In [8]:
answers = await parallel(url)
over_view = unpack(answers)
print("view == one store:", same_bits(over_view, direct))
assert same_bits(over_view, direct)
assert {a["rows"] for a in answers} == {len(documents)}

view = fse.VectorStore.open_view(stores, encoder=encoder)
print("members:", [m["rows"] for m in view.members], "source:", view.source["shard"])
view.close()
for line in stop(process, log):
    print(line)

view == one store: True
members: [32768, 49152, 38080] source: [0, 1]
INFO:     opened 81920 rows, 10 pool workers
INFO:     view 6 has 120000 rows
INFO:     all searches finished, store and pool closed


## Result

One service answered many searches at once from one store and from a view of three stores. The parallel answers had the same bits as the serial answers, and the view gave the bits of one store with the same rows. The numbers in the outputs are counts, not times. The recipe does not force a view swap or a shutdown signal while a search runs, so it does not prove that overlap. The library tests read, refresh and close with many threads at once (`tests/test_store_view_concurrent.py` and its siblings, claim class correctness).

## Before using your own service

- **Processes times workers.** Keep `processes * pool workers` at or below `usable_cpus()`. Set `PYTHON_CPU_COUNT` for each process, as this recipe does.
- **Open after the fork.** Open stores in `lifespan`, in every process. Do not use `--preload`, and do not use a store, a view or a generator from before a fork.
- **Open cost per process.** Every process opens the store itself. Write with `durability="full"` so that an open does not hash the shards.
- **Memory.** There is no shared budget. A process needs about the sum of what its running calls need. The limiter bounds the number of calls. `score` allocates its whole result, so bound its size too.
- **Backpressure.** The limiter queues requests in the event loop without bound. Add a timeout or a request limit in front of the service if clients can send faster than it answers.
- **One writer per store.** Add producers by adding stores to the view.
- **Network drives.** They are outside the crash contract. Test your own drive, and measure before you promise a speed.

## What next?

- [Vector stores in detail](../docs/architecture/store.md), with the sections "Concurrency" and "Network drives"
- [The runtime and its pool](../docs/architecture/runtime.md)
- [Document IDs](../docs/document-ids.md)
- [Rerank candidates](11-reranker-candidates.ipynb) for a second stage after the search

Back to the [recipe index](README.md) or the [main README](../README.md).

## Clean up

The helper keeps generated files in a temporary folder. Run this after you finish exploring the results.

In [9]:
store.close()
recipe.close()